In [1]:
%matplotlib widget
import importlib
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import pandas as pd
import io
import matplotlib
from scipy.optimize import fsolve
import scipy as sc
from mpmath import findroot
import mpmath
from scipy.signal import find_peaks, peak_widths
from scipy.optimize import curve_fit

os.chdir(r'C:\Users\nberr\ENS_Lyon\Thèse\Code\COMSOL\Acoustic')
import Treat_frequency_study as Treat_frequency_study
importlib.reload(Treat_frequency_study)
from Treat_frequency_study import *




import matplotlib as mpl
mpl.rcParams["font.size"] = 15

In [2]:
dir_root = r'H:\commun\Utilisateur FNO\Nathan\COMSOL\AuND - dimeres\Acoustic\Silice Thick - no layer\Frequency study'

#fname = [r'results - frequency domain - dirac excitation']
fname = [r'results - frequency domain - Heaviside excitation', r'results - frequency domain - Heaviside excitation - part 2']

param_list = ['D']
Res_simu = open_COMSOL_results_frequency_study(dir_root, fname, param_list=param_list)

fig, ax = plt.subplots()


for key in Res_simu :
    ax.plot(Res_simu[key]['freq'], Res_simu[key]['displacement'], label=key)




#ax.set_ylim(-5, 50)
#ax.legend(fontsize=10)

KeyboardInterrupt: 

In [ ]:
for key in Res_simu :
    spectra = Res_simu[key]
    
    val = spectra['displacement'].max()*0.001
    
    spectra['peaks'], _ = find_peaks( spectra['displacement'], height = val)

    #print(spectra['peaks'])

    w = peak_widths(spectra['displacement'], spectra['peaks'], rel_height=0.5)[0]   # en indices
    dx = np.mean(np.diff(spectra['freq']))
    widths = w * dx / 2

    
    # fig, ax = plt.subplots()
    # ax.scatter(spectra['freq'], spectra['displacement'], color='blue')
    # ax.scatter(spectra['freq'][spectra['peaks']], spectra['displacement'][spectra['peaks']], color='red' )

    # for k in range(len(widths)) :
   
        # ax.axvline(spectra['freq'][spectra['peaks'][k]] + widths[k]/2,  color='grey', linestyle='--' )
        # ax.axvline(spectra['freq'][spectra['peaks'][k]] - widths[k]/2,  color='grey', linestyle='--' )

In [ ]:
for key in Res_simu :
    spectra = Res_simu[key]
    
    try:
        spectra['popt'], spectra['pcov'] = fit_peaks(spectra, spectra['peaks'], key=key)
    except RuntimeError as e:
        print(f"{key} : fit non convergé ({e})")
        continue

    w = peak_widths(spectra['displacement'], spectra['peaks'], rel_height=0.5)[0]   # en indices
    dx = np.mean(np.diff(spectra['freq']))
    widths = w * dx



    fig, ax = plt.subplots()
    ax.plot(spectra['freq'], multi_lorentzian(spectra['freq'], *spectra['popt']))
    ax.scatter(spectra['freq'], spectra['displacement'], s=1, color='orange')

    ax.scatter(spectra['freq'][spectra['peaks']], spectra['displacement'][spectra['peaks']], s=10, color='red')

    
    ax.set_title(key)
    for k in range(len(widths)) :
   
        ax.axvline(spectra['freq'][spectra['peaks'][k]] + widths[k]/2,  color='grey', linestyle='--' )
        ax.axvline(spectra['freq'][spectra['peaks'][k]] - widths[k]/2,  color='grey', linestyle='--' )
        

In [ ]:
def save_data(Res_simu, dir_to_save, fname) :


    Res = {}
    Res['D (nm)'] = []
    Res['e (nm)'] = []
    Res['AR'] = []
    Res['freq (GHz)'] = []
    Res['w (GHz)'] = []
    Res['Area'] = []


    for key in Res_simu :
        spectra = Res_simu[key]
        for k in range(len(spectra['popt'])//3) :
            e=20e-9
            Res['D (nm)'].append(spectra['D'])
            Res['e (nm)'].append(e)
            Res['AR'].append(spectra['D']/e)
            Res['freq (GHz)'].append(spectra['popt'][3*k+1])
            Res['w (GHz)'].append(np.abs(spectra['popt'][3*k+2]))
            Res['Area'].append(spectra['popt'][3*k+3])
                    


    os.chdir(dir_to_save)
    df = pd.DataFrame({'D (nm)' : Res['D (nm)'],\
                       'e (nm)' : Res['e (nm)'],\
                       'AR' : Res['AR'],\
                       'freq (GHz)' : Res['freq (GHz)'],\
                       'w (GHz)' : Res['w (GHz)'],\
                       'Area' : Res['Area'],\
                        })
    df.to_csv(fname, index=False)
        

In [ ]:

def save_data_v2(Res_simu, dir_to_save, fname, e=20e-9):

    rows = []
    n_max = 0

    for key in Res_simu:
        spectra = Res_simu[key]
        popt = np.asarray(spectra['popt'])

        # popt[0] = offset, puis (f, w, Area) pour chaque lorentzienne
        n_modes = (len(popt) - 1) // 3
        n_max = max(n_max, n_modes)

        modes = []
        for k in range(n_modes):
            f    = popt[3*k + 1]
            w    = np.abs(popt[3*k + 2])
            area = popt[3*k + 3]
            modes.append((f, w, area))

        # tri par fréquence croissante
        modes.sort(key=lambda m: m[0])

        row = {
            'D (nm)': spectra['D'],
            'e (nm)': e,
            'AR': spectra['D'] / e,
        }
        for i, (f, w, area) in enumerate(modes, start=1):
            row[f'f_{i}'] = f
            row[f'w_{i}'] = w
            row[f'Area_{i}'] = area
        rows.append(row)

    # ordre des colonnes : D, e, AR, f_1, w_1, Area_1, f_2, w_2, Area_2, ...
    columns = ['D (nm)', 'e (nm)', 'AR']
    for i in range(1, n_max + 1):
        columns += [f'f_{i}', f'w_{i}', f'Area_{i}']

    df = pd.DataFrame(rows).reindex(columns=columns)
    df.to_csv(os.path.join(dir_to_save, fname), index=False)
    pass

In [ ]:
fname = 'Heaviside excitation - treated data - v2.csv'

dir_to_save = r'H:\commun\Utilisateur FNO\Nathan\COMSOL\AuND - dimeres\Acoustic\Silice Thick - no layer\Frequency study'

# save_data_v2(Res_simu, dir_to_save, fname)